# 💘 Analyse Exploratoire — Speed Dating (Projet Tinder)

**Contexte :** Tinder cherche à comprendre ce qui attire les gens les uns envers les autres, en s'appuyant sur des données collectées lors d'événements de speed dating expérimentaux (2002–2004).

Chaque ligne représente un rendez-vous de 4 minutes entre deux personnes. Les participants ont évalué leur partenaire sur 6 attributs : **Attractivité, Sincérité, Intelligence, Fun, Ambition, Intérêts communs**.

---

## 1. 📦 Importation des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['font.size'] = 12

print('Bibliotheques importees avec succes')

## 2. 📂 Chargement et exploration initiale

In [ ]:
df = pd.read_csv('Speed Dating Data.csv', encoding='latin1')

print(f'Donnees chargees avec succes')
print(f'  {df.shape[0]:,} rendez-vous enregistres')
print(f'  {df.shape[1]} colonnes')
print(f'  {df["iid"].nunique()} participants uniques')
print(f'  {df["wave"].nunique()} vagues d evenements (2002-2004)')
print(f'  Taux de match global : {df["match"].mean()*100:.1f}%')
df.head(3)

In [ ]:
# Valeurs manquantes (colonnes avec plus de 10% de NaN)
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_high = missing[missing > 10]
print(f'{len(missing_high)} colonnes avec plus de 10% de valeurs manquantes')
print(missing_high.head(15).round(1).to_string())

## 3. 🧹 Nettoyage et préparation

In [ ]:
# Selection des colonnes cles pour cette analyse
cols = [
    'iid', 'gender', 'match', 'dec', 'dec_o', 'order', 'samerace',
    'int_corr', 'imprace', 'age', 'age_o', 'race', 'race_o',
    # Notes donnees au partenaire (le soir du rendez-vous)
    'attr', 'sinc', 'intel', 'fun', 'amb', 'shar',
    # Notes recues du partenaire
    'attr_o', 'sinc_o', 'intel_o', 'fun_o', 'amb_o', 'shar_o',
    # Ce que les participants declarent rechercher (avant l evenement, 100 pts a repartir)
    'attr1_1', 'sinc1_1', 'intel1_1', 'fun1_1', 'amb1_1', 'shar1_1',
    # Auto-evaluation avant l evenement (sur 10)
    'attr3_1', 'sinc3_1', 'intel3_1', 'fun3_1', 'amb3_1',
]

df_c = df[cols].copy()
df_c['gender_label'] = df_c['gender'].map({0: 'Femme', 1: 'Homme'})

print(f'Dataset nettoye : {df_c.shape[0]:,} lignes x {df_c.shape[1]} colonnes')
df_c.describe().round(2)

## 4. 📊 Statistiques Descriptives

In [ ]:
attrs       = ['attr',   'sinc',       'intel',         'fun', 'amb',      'shar']
attr_labels = ['Attractivite','Sincerite','Intelligence','Fun','Ambition','Interets communs']

# Taux de match et de decision positive
match_rate = df_c['match'].mean() * 100
dec_h = df_c[df_c['gender_label']=='Homme']['dec'].mean() * 100
dec_f = df_c[df_c['gender_label']=='Femme']['dec'].mean() * 100

print('='*50)
print(f'  Taux de match global       : {match_rate:.1f}%')
print(f'  Decision positive (Hommes) : {dec_h:.1f}%')
print(f'  Decision positive (Femmes) : {dec_f:.1f}%')
print('='*50)

# Notes moyennes par attribut
stats = df_c[attrs].agg(['mean','median','std']).T
stats.index = attr_labels
stats.columns = ['Moyenne','Mediane','Ecart-type']
print('\nNotes attribuees aux partenaires (sur 10) :')
print(stats.round(2).to_string())

## 5. 📈 Visualisations

### 5.1 — Distribution des matchs et décisions par genre

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart : matchs
n_match    = int(df_c['match'].sum())
n_no_match = len(df_c) - n_match
axes[0].pie(
    [n_no_match, n_match],
    labels=[f'Pas de match ({n_no_match:,})', f'Match ({n_match:,})'],
    colors=['#E74C3C', '#2ECC71'],
    autopct='%1.1f%%', startangle=90,
    textprops={'fontsize': 12}
)
axes[0].set_title('Distribution globale des matchs', fontsize=14, fontweight='bold')

# Barres : decision positive par genre
dec_gender = df_c.groupby('gender_label')['dec'].mean() * 100
bars = axes[1].bar(dec_gender.index, dec_gender.values,
                   color=['#FF69B4', '#4169E1'], width=0.45, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, dec_gender.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', fontsize=13, fontweight='bold')
axes[1].set_title('Taux de decision positive par genre', fontsize=14, fontweight='bold')
axes[1].set_ylabel('% de "Oui" pour un 2eme RDV')
axes[1].set_ylim(0, 60)

plt.tight_layout()
plt.show()

print("""
Interpretation :
Seulement 16.5% des speed dates aboutissent a un match mutuel.
Les hommes disent oui dans ~45% des cas, contre ~35% pour les femmes.
Les femmes sont significativement plus selectives que les hommes.
""")

### 5.2 — Quels attributs influencent le plus les matchs ?

In [ ]:
match_means    = df_c[df_c['match']==1][attrs].mean()
no_match_means = df_c[df_c['match']==0][attrs].mean()

x     = np.arange(len(attrs))
width = 0.35
fig, ax = plt.subplots(figsize=(13, 6))

b1 = ax.bar(x - width/2, match_means.values, width,
            label='Match', color='#2ECC71', edgecolor='white', linewidth=1.2)
b2 = ax.bar(x + width/2, no_match_means.values, width,
            label='Pas de match', color='#E74C3C', edgecolor='white', linewidth=1.2)

for bar in b1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
            f'{bar.get_height():.1f}', ha='center', fontsize=10, fontweight='bold', color='#27AE60')
for bar in b2:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
            f'{bar.get_height():.1f}', ha='center', fontsize=10, color='#C0392B')

ax.set_xticks(x)
ax.set_xticklabels(attr_labels, fontsize=11)
ax.set_ylabel('Note moyenne (sur 10)')
ax.set_ylim(0, 10)
ax.set_title('Notes moyennes des attributs : Match vs Pas de match', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print("""
Interpretation :
Tous les attributs sont notes plus haut quand un match se produit.
L attractivite montre l ecart le plus important entre match et non-match,
suivie du fun et des interets communs.
La sincerite et l intelligence ont un impact plus limite sur la decision finale.
""")

### 5.3 — Attributs les moins désirables : différence Hommes / Femmes

In [ ]:
# attr1_1 a shar1_1 = importance accordee a chaque attribut AVANT le rendez-vous (100 pts a repartir)
imp_cols   = ['attr1_1','sinc1_1','intel1_1','fun1_1','amb1_1','shar1_1']
imp_labels = ['Attractivite','Sincerite','Intelligence','Fun','Ambition','Interets communs']

imp_gender = df_c.groupby('gender_label')[imp_cols].mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
palette   = {'Femme': '#FF69B4', 'Homme': '#4169E1'}

for idx, (gender, row) in enumerate(imp_gender.iterrows()):
    vals        = row.values
    order_idx   = np.argsort(vals)
    sorted_lbls = [imp_labels[i] for i in order_idx]
    sorted_vals = vals[order_idx]

    bars = axes[idx].barh(sorted_lbls, sorted_vals, color=palette[gender], alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, sorted_vals):
        axes[idx].text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
                       f'{val:.1f}', va='center', fontsize=11, fontweight='bold')
    axes[idx].set_title(f'{gender}s - Importance accordee aux criteres', fontsize=13, fontweight='bold')
    axes[idx].set_xlabel('Points attribues (sur 100 au total)')
    axes[idx].set_xlim(0, 35)

plt.suptitle('Ce que chaque genre recherche chez un partenaire (avant le RDV)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("""
Interpretation :
Les hommes accordent plus de points a l attractivite physique que les femmes.
Les femmes valorisent davantage l intelligence, la sincerite et l ambition.
L attribut le MOINS desire pour les deux genres est l ambition,
suggerant qu une personnalite ambitieuse est percue comme moins romantique.
""")

### 5.4 — L'attractivité : importance déclarée vs impact réel sur le match

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Gauche : importance declaree
imp_means  = df_c[imp_cols].mean().values
bar_colors = ['#E74C3C' if i==0 else '#BDC3C7' for i in range(len(imp_cols))]
axes[0].bar(imp_labels, imp_means, color=bar_colors, edgecolor='white', linewidth=1.2)
axes[0].set_title('Importance declaree avant le RDV', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Points moyens attribues (sur 100)')
axes[0].set_xticklabels(imp_labels, fontsize=9, rotation=15)

# Droite : correlation reelle avec le match
correlations = df_c[attrs + ['match']].corr()['match'].drop('match')
bar_colors2  = ['#E74C3C' if i==0 else '#BDC3C7' for i in range(len(attrs))]
axes[1].bar(attr_labels, correlations.values, color=bar_colors2, edgecolor='white', linewidth=1.2)
axes[1].set_title('Correlation reelle avec le match', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Coefficient de correlation de Pearson')
axes[1].set_xticklabels(attr_labels, fontsize=9, rotation=15)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')

plt.suptitle('Attractivite (rouge) : discours vs realite', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("""
Interpretation :
L attractivite est l attribut le plus correle avec l obtention d un match.
Les participants la declarent importante, et leur comportement reel le confirme.
En revanche, la sincerite et l intelligence sont citees comme importantes
mais ont une correlation bien plus faible avec le match reel -
ce qui revele un biais de desirabilite sociale : on dit ce qu on devrait valoriser,
pas necessairement ce qui guide nos decisions.
""")

### 5.5 — Intérêts communs vs Origine raciale commune

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gauche : distribution int_corr selon le match
df_ic = df_c[['int_corr','match']].dropna()
m_yes = df_ic[df_ic['match']==1]['int_corr']
m_no  = df_ic[df_ic['match']==0]['int_corr']

axes[0].hist(m_no,  bins=25, alpha=0.55, color='#E74C3C', label='Pas de match', density=True)
axes[0].hist(m_yes, bins=25, alpha=0.55, color='#2ECC71', label='Match',      density=True)
axes[0].axvline(m_no.mean(),  color='#C0392B', linestyle='--', linewidth=2,
                label=f'Moy. non-match ({m_no.mean():.2f})')
axes[0].axvline(m_yes.mean(), color='#27AE60', linestyle='--', linewidth=2,
                label=f'Moy. match ({m_yes.mean():.2f})')
axes[0].set_title('Similarite des interets (int_corr) selon le match', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Correlation des interets entre partenaires')
axes[0].set_ylabel('Densite')
axes[0].legend(fontsize=9)

# Droite : taux de match selon origine raciale commune
race_match = df_c.groupby('samerace')['match'].mean() * 100
race_labels = ['Races differentes', 'Meme race']
bars = axes[1].bar(race_labels, race_match.values,
                   color=['#F39C12','#8E44AD'], width=0.45, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, race_match.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{val:.1f}%', ha='center', fontsize=13, fontweight='bold')
axes[1].set_title('Taux de match : meme race vs races differentes', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Taux de match (%)')
axes[1].set_ylim(0, 25)

plt.suptitle('Interets communs vs Origine raciale commune', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"""
Interpretation :
La correlation des interets est legerement plus elevee chez les couples qui matchent
({m_yes.mean():.2f} vs {m_no.mean():.2f} pour les non-matchs), mais la difference est moderee.

Partager la meme origine raciale augmente legerement le taux de match
({race_match[1]:.1f}% vs {race_match[0]:.1f}%), mais cet effet reste inferieur
a celui de l attractivite ou du fun.
Les interets communs semblent donc plus determinants que l origine raciale partagee.
""")

### 5.6 — Les gens peuvent-ils prédire leur propre valeur sur le marché des rencontres ?

In [ ]:
# Auto-evaluation (attr3_1 = comment je me note moi-meme, avant le RDV, sur 10)
# Notes recues du partenaire le soir du rendez-vous (attr_o, fun_o, ...)
self_cols = ['attr3_1','sinc3_1','intel3_1','fun3_1','amb3_1']
recv_cols = ['attr_o', 'sinc_o', 'intel_o', 'fun_o', 'amb_o']
comp_lbls = ['Attractivite','Sincerite','Intelligence','Fun','Ambition']

self_means = df_c[self_cols].mean().values
recv_means = df_c[recv_cols].mean().values

x     = np.arange(len(comp_lbls))
width = 0.35
fig, ax = plt.subplots(figsize=(13, 6))

b1 = ax.bar(x - width/2, self_means, width,
            label='Auto-evaluation (avant le RDV)', color='#3498DB', alpha=0.85, edgecolor='white')
b2 = ax.bar(x + width/2, recv_means, width,
            label='Notes recues des partenaires', color='#E67E22', alpha=0.85, edgecolor='white')

for bar in b1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.08,
            f'{bar.get_height():.1f}', ha='center', fontsize=10, fontweight='bold', color='#2980B9')
for bar in b2:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.08,
            f'{bar.get_height():.1f}', ha='center', fontsize=10, fontweight='bold', color='#D35400')

ax.set_xticks(x)
ax.set_xticklabels(comp_lbls, fontsize=12)
ax.set_ylabel('Score moyen (sur 10)')
ax.set_ylim(0, 10)
ax.set_title('Auto-evaluation vs Notes reellement recues des partenaires', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print("""
Interpretation :
Les participants se surevaluent systematiquement sur tous les attributs.
L ecart est particulierement marque pour l attractivite et le fun :
les individus se percoivent comme bien plus attractifs que ce que leurs partenaires en pensent.
Ce biais de positivite montre que les gens ont une perception deformee
de leur propre valeur sur le marche des rencontres.
""")

### 5.7 — Vaut-il mieux être le premier ou le dernier speed date de la soirée ?

In [ ]:
order_match = df_c.groupby('order')['match'].mean() * 100
smooth = order_match.rolling(window=3, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 6))

ax.bar(order_match.index, order_match.values, color='#BDC3C7', alpha=0.5, label='Taux brut')
ax.plot(smooth.index, smooth.values, color='#9B59B6', linewidth=2.5,
        marker='o', markersize=5, markerfacecolor='white', markeredgewidth=2,
        label='Tendance (moy. mobile 3)')
ax.fill_between(smooth.index, smooth.values, alpha=0.1, color='#9B59B6')

first_v = order_match.iloc[0]
last_v  = order_match.iloc[-1]
first_i = order_match.index[0]
last_i  = order_match.index[-1]

ax.scatter([first_i, last_i], [first_v, last_v], color=['#2ECC71','#E74C3C'], s=180, zorder=5)
ax.annotate(f'1er RDV : {first_v:.1f}%', xy=(first_i, first_v),
            xytext=(first_i+1, first_v+3), fontsize=11, color='#27AE60', fontweight='bold')
ax.annotate(f'Dernier RDV : {last_v:.1f}%', xy=(last_i, last_v),
            xytext=(last_i-5, last_v+3), fontsize=11, color='#C0392B', fontweight='bold')

ax.set_xlabel("Ordre du rendez-vous dans la soiree", fontsize=12)
ax.set_ylabel("Taux de match (%)", fontsize=12)
ax.set_title("Taux de match selon l ordre du speed date dans la soiree", fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("""
Interpretation :
Le taux de match varie selon la position dans la soiree.
On observe un effet de contraste : chaque partenaire est juge par rapport aux precedents.
Les premiers rendez-vous beneficient d un jugement plus neutre et sans comparaison.
En fin de soiree, la fatigue de decision peut reduire la tendance a accepter un second RDV.
""")

### 5.8 — Heatmap des corrélations entre attributs et match

In [ ]:
corr_cols   = attrs + ['match']
corr_labels = attr_labels + ['Match']

corr_matrix = df_c[corr_cols].corr()
corr_matrix.index   = corr_labels
corr_matrix.columns = corr_labels

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix, mask=mask,
    annot=True, fmt='.2f', cmap='RdYlGn', center=0,
    square=True, linewidths=0.5, ax=ax,
    vmin=-0.3, vmax=0.7
)
ax.set_title('Matrice de correlation - Attributs et Match', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("""
Interpretation :
L attractivite est l attribut le plus fortement correle avec le match.
On observe un fort effet de halo : les attributs sont tres correles entre eux.
Un partenaire percu comme attractif est aussi percu comme fun, intelligent et sincere.
La premiere impression physique influence donc toute la perception du partenaire.
""")

## 6. 📝 Conclusion Générale

### Ce que les données révèlent sur l'attraction humaine

| Question | Réponse tirée des données |
|---|---|
| **Attribut le moins désiré ?** | L'ambition — pour les deux genres |
| **Différence Hommes / Femmes ?** | Les hommes priorisent l'attractivité, les femmes l'intelligence et la sincérité |
| **Attractivité : discours vs réalité ?** | Le discours est confirmé — l'attractivité est bien le 1er facteur de match réel |
| **Intérêts communs vs origine raciale ?** | Les intérêts communs ont plus d'impact sur le match que l'origine raciale partagée |
| **Auto-évaluation fiable ?** | Non — les participants se surévaluent systématiquement par rapport aux notes reçues |
| **Ordre du RDV dans la soirée ?** | Légère avantage pour les premiers RDV ; la fatigue de décision pénalise les derniers |

---

### 💡 Recommandation stratégique pour Tinder

> Pour augmenter le taux de correspondances sur l'application, Tinder devrait :
>
> 1. **Mettre l'accent sur les photos de profil** — l'attractivité reste le premier déclencheur d'intérêt.
> 2. **Valoriser les intérêts communs** dans le matching — plus efficace que les critères démographiques.
> 3. **Réduire le biais de surévaluation** en fournissant aux utilisateurs un feedback honnête sur leur profil.
> 4. **Adapter les recommandations par genre** — les femmes cherchent des profils équilibrés (intelligence + sincérité), les hommes sont plus sensibles à l'aspect physique.